<a href="https://colab.research.google.com/github/aayushchourasia123/Machine-Learning/blob/main/automatically%20select%20imputer%20parameter/automatically_select_imputer_parameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [19]:
df=pd.read_csv('train.csv')

In [20]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [21]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [22]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [23]:
x=df.drop(columns=['Survived'])
y=df['Survived']

In [24]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

In [25]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [26]:
numeric_feature=['Age','Fare']
numeric_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_feature=['Embarked','Sex']
categorical_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

In [27]:
preprocessing=ColumnTransformer(transformers=[
    ('num',numeric_transformer,numeric_feature),
    ('cat',categorical_transformer,categorical_feature)
])

In [28]:
clf=Pipeline(steps=[
    ('preprocessor',preprocessing),
    ('classifier',LogisticRegression())
])

In [30]:
from sklearn import set_config
set_config(display='diagram')
clf

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Embarked', 'Sex'])])),
                ('classifier', LogisticRegression())])

In [35]:
from sklearn.model_selection import GridSearchCV
param_grid={
    'classifier__C':[0.1,1.0,10,100],
    'preprocessor__num__imputer__strategy':['mean','median'],
    'preprocessor__cat__imputer__strategy':['most_frequent','constant']
}
grid_search=GridSearchCV(clf,param_grid,cv=10)

In [36]:
grid_search.fit(x_train,y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'Fare']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Embarked',
                                                                          'Sex'])])),
                                       ('classifier', LogisticRegression())]),
             param_grid={'classifier__C': [0.1, 1.0, 10, 100],
                         'preprocessor__cat__imputer__strategy': ['most_frequent',
                                                                  'constant'],
                         'preprocessor__num__imputer__strategy': ['mean',
                                                                  'median']})

In [37]:
print('best params')
print(grid_search.best_params_)

best params
{'classifier__C': 0.1, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}


In [38]:
print('internal cv score', grid_search.best_score_)

internal cv score 0.7837245696400627


In [46]:
cv_result=pd.DataFrame(grid_search.cv_results_)
cv_result=cv_result.sort_values('rank_test_score')
cv_result[['param_classifier__C','param_preprocessor__cat__imputer__strategy','param_preprocessor__num__imputer__strategy','mean_test_score']]


,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,mean_test_score
0,0.1,most_frequent,mean,0.783725
1,0.1,most_frequent,median,0.783725
2,0.1,constant,mean,0.783725
3,0.1,constant,median,0.783725
4,1.0,most_frequent,mean,0.782316
5,1.0,most_frequent,median,0.782316
6,1.0,constant,mean,0.782316
7,1.0,constant,median,0.782316
8,10.0,most_frequent,mean,0.782316
9,10.0,most_frequent,median,0.782316


In [44]:
cv_result

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,params,split0_test_score,split1_test_score,...,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.032240,0.013242,0.015199,0.006683,0.1,most_frequent,mean,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
1,0.063039,0.070834,0.016698,0.008508,0.1,most_frequent,median,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
2,0.027499,0.012494,0.013324,0.007825,0.1,constant,mean,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
3,0.094370,0.069300,0.026147,0.014985,0.1,constant,median,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
4,0.087946,0.039795,0.020887,0.008816,1.0,most_frequent,mean,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
5,0.063200,0.034240,0.019154,0.004780,1.0,most_frequent,median,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
6,0.050848,0.028523,0.018162,0.009237,1.0,constant,mean,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
7,0.023437,0.002473,0.009939,0.000460,1.0,constant,median,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
8,0.021012,0.001478,0.010350,0.001480,10.0,most_frequent,mean,"{'classifier__C': 10, 'preprocessor__cat__impu...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
9,0.020973,0.001863,0.009324,0.000501,10.0,most_frequent,median,"{'classifier__C': 10, 'preprocessor__cat__impu...",0.805556,0.75,...,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
